# Лабораторная 2 — спайки AI-пайплайна

Одноразовый воспроизводимый эксперимент без API-ключа. Проверяем два вопроса: помещается ли целевой документ в прямой промпт и отсекают ли детерминированные проверки структурно неверные/неподтверждённые карточки. Это не production-код и не доказательство качества конкретной LLM.

## Спайк 1. Контекстный бюджет без RAG

Внутренний лимит намеренно меньше контекстного окна провайдера: он ограничивает стоимость и задержку.

In [1]:
INTERNAL_LIMIT = 100_000
PROMPT_AND_SCHEMA = 2_000
EXPECTED_OUTPUT = 4_000
SAFETY_MARGIN = 0.20

def budget(document_tokens):
    subtotal = document_tokens + PROMPT_AND_SCHEMA + EXPECTED_OUTPUT
    total = int(subtotal * (1 + SAFETY_MARGIN))
    return {
        'document_tokens': document_tokens,
        'total_with_margin': total,
        'fits': total <= INTERNAL_LIMIT,
        'remaining': INTERNAL_LIMIT - total,
    }

scenarios = {
    'typical_pdf': 25_000,
    'mvp_boundary': 60_000,
    'textbook_or_bad_upload': 95_000,
}
budget_results = {name: budget(tokens) for name, tokens in scenarios.items()}
budget_results


{'typical_pdf': {'document_tokens': 25000, 'total_with_margin': 37200, 'fits': True, 'remaining': 62800},
 'mvp_boundary': {'document_tokens': 60000, 'total_with_margin': 79200, 'fits': True, 'remaining': 20800},
 'textbook_or_bad_upload': {'document_tokens': 95000, 'total_with_margin': 121200, 'fits': False, 'remaining': -21200}}

In [2]:
assert budget_results['typical_pdf']['fits']
assert budget_results['mvp_boundary']['fits']
assert not budget_results['textbook_or_bad_upload']['fits']
print('PASS: типичный PDF и граница MVP проходят; слишком большой документ отклоняется.')


PASS: типичный PDF и граница MVP проходят; слишком большой документ отклоняется.


## Спайк 2. Schema + grounding checks

Фикстуры имитируют результаты модели. Валидатор проверяет обязательные поля, количество и уникальность карточек, существование страницы и дословное вхождение нормализованной цитаты.

In [3]:
import copy
import re

pages = {
    1: 'Кэш процессора — быстрая память между процессором и оперативной памятью. Он хранит копии часто используемых данных.',
    2: 'Кэш делится на уровни L1, L2 и L3. Каждый следующий уровень медленнее, но больше по объёму.',
    3: 'Промах кэша происходит, когда нужных данных в кэше нет, и процессор обращается к оперативной памяти.',
}

def norm(text):
    return re.sub(r'\s+', ' ', text).strip().lower()

def validate_card_set(payload, source_pages):
    errors = []
    if payload.get('status') != 'ready':
        errors.append('status must be ready')
    cards = payload.get('cards')
    if not isinstance(cards, list) or not 5 <= len(cards) <= 10:
        return errors + ['cards count must be 5..10']
    required = {'card_id', 'question', 'reference_answer', 'source_page', 'source_quote', 'difficulty'}
    seen_ids = set()
    for index, card in enumerate(cards):
        missing = required - set(card)
        if missing:
            errors.append(f'card[{index}] missing: {sorted(missing)}')
            continue
        if card['card_id'] in seen_ids:
            errors.append(f'card[{index}] duplicate card_id')
        seen_ids.add(card['card_id'])
        page_text = source_pages.get(card['source_page'])
        if page_text is None:
            errors.append(f'card[{index}] source page does not exist')
        elif norm(card['source_quote']) not in norm(page_text):
            errors.append(f'card[{index}] quote is not grounded')
    return errors


In [4]:
valid = {
    'status': 'ready',
    'document_id': 'doc_cache',
    'title': 'Кэш процессора',
    'cards': [
        {'card_id': 'card_1', 'question': 'Где расположен кэш?', 'reference_answer': 'Между процессором и RAM.', 'source_page': 1, 'source_quote': 'быстрая память между процессором и оперативной памятью', 'difficulty': 'basic'},
        {'card_id': 'card_2', 'question': 'Что хранит кэш?', 'reference_answer': 'Копии часто используемых данных.', 'source_page': 1, 'source_quote': 'Он хранит копии часто используемых данных', 'difficulty': 'basic'},
        {'card_id': 'card_3', 'question': 'Какие уровни есть?', 'reference_answer': 'L1, L2 и L3.', 'source_page': 2, 'source_quote': 'Кэш делится на уровни L1, L2 и L3', 'difficulty': 'basic'},
        {'card_id': 'card_4', 'question': 'Как меняется объём?', 'reference_answer': 'Следующий уровень больше.', 'source_page': 2, 'source_quote': 'Каждый следующий уровень медленнее, но больше по объёму', 'difficulty': 'intermediate'},
        {'card_id': 'card_5', 'question': 'Что происходит при промахе?', 'reference_answer': 'Процессор обращается к RAM.', 'source_page': 3, 'source_quote': 'процессор обращается к оперативной памяти', 'difficulty': 'intermediate'},
    ],
}
missing_quote = copy.deepcopy(valid)
del missing_quote['cards'][0]['source_quote']
hallucinated_quote = copy.deepcopy(valid)
hallucinated_quote['cards'][1]['source_page'] = 99
hallucinated_quote['cards'][2]['source_quote'] = 'Кэш полностью заменяет оперативную память.'


In [5]:
fixtures = {
    'valid': valid,
    'missing_quote': missing_quote,
    'hallucinated_quote': hallucinated_quote,
}
results = {name: validate_card_set(payload, pages) for name, payload in fixtures.items()}
assert results['valid'] == []
assert results['missing_quote']
assert results['hallucinated_quote']
for name, errors in results.items():
    print(f'{name}:', 'ACCEPTED' if not errors else f'REJECTED — {errors}')


valid: ACCEPTED
missing_quote: REJECTED — ["card[0] missing: ['source_quote']"]
hallucinated_quote: REJECTED — ['card[1] source page does not exist', 'card[2] quote is not grounded']


## Вывод

Прямой промпт достаточен по размеру для границ MVP. Строгая схема необходима, но недостаточна: после неё нужен отдельный grounding check. Фактическое качество Gemini, latency и обработка реальных PDF этим спайком не проверены.